# day-05-prompting-techniques — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [9]:
# ---- Solution 1 ----
def build_mixed_prompt(examples, query):
    lines = ["Classify the review.\n"]
    for i, (t, y) in enumerate(examples):
        fmt = f"Label: {y}" if i % 2 == 0 else f"Sentiment - {y}"
        lines.append(f"Review: {t}\n{fmt}\n")
    lines.append(f"Review: {query}\nLabel:")
    return "\n".join(lines)
print(build_mixed_prompt(labeled[:4], "great phone"))
print("\n# A real model sees two competing output templates and picks unpredictably per call,")
print("# or blends them. Downstream parsing breaks. Keep EVERY example's format identical to")
print("# the format you want back.")

Classify the review.

Review: works great out of the box
Label: positive

Review: exceeded my expectations
Sentiment - positive

Review: high quality, would buy again
Label: positive

Review: the fabric feels cheap and thin
Sentiment - negative

Review: great phone
Label:

# A real model sees two competing output templates and picks unpredictably per call,
# or blends them. Downstream parsing breaks. Keep EVERY example's format identical to
# the format you want back.


In [10]:
# ---- Solution 2 ----
def few_shot_random(text, k=3):
    idx = rng.choice(len(labeled), size=k, replace=False)
    sims_labels = [Y[i] for i in idx]
    return Counter(sims_labels).most_common(1)[0][0]

rand_acc = np.mean([[few_shot_random(t) == y for t, y in reviews] for _ in range(20)])
sim_acc  = np.mean([few_shot_classify(t, 3) == y for t, y in reviews])
print(f"S2: random-3 examples  ~{rand_acc:.2f}   most-similar-3 examples {sim_acc:.2f}")

S2: random-3 examples  ~0.50   most-similar-3 examples 0.62


In [11]:
# ---- Solution 3 ----
def direct_tokens(e):
    return ntok(str(direct_answer(e)))
def cot_tokens(e):
    ans, steps = cot_answer(e)
    return ntok("\n".join(steps) + f"\nfinal: {ans}")
d = np.mean([direct_tokens(e) for e in exprs])
c = np.mean([cot_tokens(e) for e in exprs])
print(f"S3: direct ~{d:.1f} output tokens, CoT ~{c:.1f}  -> CoT tax ~{c/d:.1f}x")

S3: direct ~1.1 output tokens, CoT ~36.9  -> CoT tax ~32.2x


In [12]:
# ---- Solution 4 ----
trivial = "5 + 2"
print("direct:", direct_answer(trivial), " (may be off by the injected guess noise)")
ans, steps = cot_answer(trivial)
print("CoT:", ans, "via", steps, f"-> {ntok(chr(10).join(steps))} tokens for a 1-step problem")
print("S4: same answer quality, more tokens + latency. CoT is overhead when depth <= 1.")

direct: 5  (may be off by the injected guess noise)
CoT: 7 via ['start: 5', '+ 2 -> 7'] -> 11 tokens for a 1-step problem
S4: same answer quality, more tokens + latency. CoT is overhead when depth <= 1.


In [13]:
# ---- Solution 5 ----
import math
n = math.log(0.25) / math.log(0.92)
print(f"S5: influence < 0.25 after {n:.1f} turns; check: {0.92**17:.3f} at 17, {0.92**16:.3f} at 16")

S5: influence < 0.25 after 16.6 turns; check: 0.242 at 17, 0.263 at 16


### Solution 6

- **(a) 20 invoice docs → JSON:** few-shot with 2–3 worked examples showing the exact JSON
  schema. Zero-shot might do it; few-shot locks the format. No fine-tuning needed at this scale.
- **(b) 2M docs/month, must be exact:** fine-tune on labeled extractions + constrained/JSON
  decoding. Per-call few-shot tokens × 2M is expensive and still has a failing tail; baking it
  into weights gives a short prompt and higher reliability.
- **(c) 500-page handbook:** RAG. The knowledge isn't in the model; chunk + embed the handbook,
  retrieve relevant passages, answer from them with citations. (Weeks 5–6.)
- **(d) Always respond as "Captain Nemo":** system prompt is enough for a demo; if it must hold
  across very long conversations or thousands of sessions without drift, a light fine-tune on
  in-character dialogue removes the drift and the per-call token cost.

### Answer key
1. The model predicts the next token from training-data patterns. Instruction-following exists
   only because instruction tuning added many `instruction → good response` examples;
   "compliance" is just the most probable continuation given that training.
2. It's the first block of the single flattened token sequence. Because attention has a
   recency bias and later tokens accumulate, its relative influence on the next-token
   prediction decays as the conversation grows — hence drift.
3. Any three of: number of examples, similarity of examples to the query, label balance,
   format consistency, ordering (recency).
4. It converts a deep sequential computation into a longer sequence of tokens, and the model
   gets a fixed compute budget *per token*, so more tokens = more total computation.
5. Easy tasks (adds latency/cost for no gain); tasks where the model's first instinct is
   correct but it "reasons itself out of it"; very tight token budgets.
6. Prompting typically plateaus around 95–99%; the long tail of malformed outputs remains.
   Fix: fine-tune for the format plus constrained decoding / a tool schema that makes invalid
   output impossible.
7. Effort low→high: system prompt < few-shot < RAG < fine-tuning. System prompt and few-shot
   *cannot* solve it (the facts aren't available). RAG solves it by putting the API docs in
   context at query time; fine-tuning solves it by baking them into weights (more effort,
   staler). RAG is the usual answer.